# DRAM Bender Python tutorial

DRAM Bender provides both Python and C++ APIs for constructing and executing precise DRAM command programs. This notebook uses the Python API to explain the common program model, target-aware built-in programs, software debugging tools, hardware execution, custom programs, and JIT templates. C++ examples are linked at the end.

The default notebook run is software-only. It builds and inspects real DRAM Bender programs without opening an FPGA. Hardware execution requires an explicit environment setting and a complete PCI BDF.

## Contents

1. Setup, board configurations, and targets
2. Built-in read/write programs
3. Program inspection, dry-run, and timing traces
4. Hardware execution and recovery
5. RowHammer with built-in programs
6. Custom programs with ProgramBuilder
7. JIT-compiled program templates
8. Advanced RowHammer composition
9. Retention experiments
10. Next steps


## DRAM Bender execution model

A DRAM Bender program combines scalar/control instructions with packed DRAM commands.

- A fabric instruction occupies one 64-bit instruction word.
- A DRAM instruction packs four 16-bit mini-operations into one fabric word.
- Scalar arithmetic and data operations take one fabric cycle.
- Conditional branches and jumps resolve in six fabric cycles.
- LABEL is an assembler marker and does not emit an instruction.
- SLEEP(N) consumes N fabric cycles. Small sleeps lower to explicit all-NOP DRAM words; larger sleeps use the sleep instruction.

ProgramBuilder copies the target BoardConfig's DRAM command-slot duration to the FinalProgram, which the software VM uses by default. The VM defaults to four DRAM mini-slots per fabric cycle, matching every maintained board configuration. Both values can be overridden when modeling a bitstream with different timing or command packing. Always use the requirements of the actual memory device, speed bin, and bitstream when deciding legal command spacing.

Two ProgramBuilder methods issue DRAM commands:

- <code>DRAM(op0, op1, op2, op3)</code> exposes the exact four-slot layout.
- <code>DRAMSEQ(..., ALIGN())</code> lays out a timed sequence whose delays are expressed in mini-slots.

DRAM Bender does not enforce memory timing constraints. The software VM helps inspect command spacing, but it does not model RTL, DRAM data, analog behavior, or every JEDEC timing rule.


## 1. Setup, board configurations, and targets

Create the development environment before opening this notebook:

    bash setup_venv.sh

Then select the repository virtual environment as the Jupyter kernel.

The hardware cells assume a controlled DRAM experiment system and modify the selected DRAM rows. Verify the programmed bitstream, memory geometry, and target addresses before enabling hardware execution.

The endpoint selector has two parts:

- <code>PCI_BDF</code> identifies the PCI function, for example <code>0000:01:00.0</code>.
- <code>XDMA_CHANNEL</code> identifies an independent controller endpoint on that function.

The API resolves these values to the matching H2C and C2H device nodes. Names such as <code>/dev/xdma0_h2c_0</code> contain a probe-order number that can change after reboot or driver reload, so they are not stable board identifiers.


In [ ]:
import os
import sys

import numpy as np

import drambender
from drambender.api import (
    DDR4Target,
    FinalProgram,
    HBM2U55Target,
    HostInterface,
    ProgramBuilder,
    board_configs,
    open_board,
    program_template,
)
from drambender.api.program.instructions import *

assert sys.version_info >= (3, 10)
print(f"Python: {sys.executable}")
print(f"drambender: {drambender.__file__}")

# Hardware is opt-in so this notebook also runs without an FPGA. Set these
# before starting Jupyter, or edit the values here.
RUN_HARDWARE = os.environ.get("DRAMBENDER_RUN_HARDWARE") == "1"
ROW_MAPPING = os.environ.get("DRAMBENDER_ROW_MAPPING", "linear")
PCI_BDF = os.environ.get("DRAMBENDER_PCI_BDF", "")
XDMA_CHANNEL = int(os.environ.get("DRAMBENDER_XDMA_CHANNEL", "0"))

if RUN_HARDWARE and not PCI_BDF:
    raise RuntimeError(
        "DRAMBENDER_RUN_HARDWARE=1 requires DRAMBENDER_PCI_BDF=dddd:bb:ss.f"
    )
print(f"Hardware execution: {RUN_HARDWARE}")
print(f"Row mapping: {ROW_MAPPING}")
if RUN_HARDWARE:
    print(f"Endpoint: {PCI_BDF}, XDMA channel {XDMA_CHANNEL}")


### Memory targets

The same program-building workflow supports both maintained memory families:

- <code>DDR4Target</code> describes external DDR4 on Alveo U200.
- <code>HBM2U50Target</code> describes HBM2 on Alveo U50.
- <code>HBM2U55Target</code> describes HBM2 on Alveo U55C.

<code>HBM2Target</code> is their abstract base and cannot be instantiated directly. DDR4Target has defaults for the common maintained configuration. Explicit values are used below so that the geometry is visible. Verify them against the module and bitstream.

Every target exposes its immutable API assumptions as <code>target.board_config</code>. The same built-in objects are available as <code>board_configs.U200</code>, <code>board_configs.U50</code>, and <code>board_configs.U55C</code>. They describe the expected instruction and readback capacities, command timing, HBM topology, and optional board features. These values are not discovered from the programmed FPGA image.

For HBM2, <code>channel</code>, <code>pseudo_channel</code>, and <code>sid</code> are memory coordinates. They are distinct from <code>XDMA_CHANNEL</code>, which selects the host transport endpoint.


In [ ]:
CACHELINES_PER_ROW = 128
WORDS_PER_CACHELINE = 16
COLUMN_STRIDE = 8

DDR4_TARGET = DDR4Target(
    cachelines_per_row=CACHELINES_PER_ROW,
    column_stride=COLUMN_STRIDE,
    words_per_cacheline=WORDS_PER_CACHELINE,
    rank=0,
)

HBM2_TARGET = HBM2U55Target(
    channel=0,
    pseudo_channel=0,
    sid=0,  # Set channel, pseudo-channel, and SID from the bitstream docs.
    columns_per_row=32,
    column_stride=1,
    words_per_cacheline=16,
)

DDR4_ROW_BYTES = (
    DDR4_TARGET.cachelines_per_row
    * DDR4_TARGET.words_per_cacheline
    * 4
)
HBM2_ROW_BYTES = HBM2_TARGET.receive_bytes_per_row

print(f"DDR4 row readback: {DDR4_ROW_BYTES} bytes")
print(f"HBM2 row readback: {HBM2_ROW_BYTES} bytes")
assert DDR4_TARGET.board_config is board_configs.U200
assert HBM2_TARGET.board_config is board_configs.U55C
print(DDR4_TARGET.board_config.summary())
print(HBM2_TARGET.board_config.summary())


## 2. Built-in read/write programs

<code>drambender.builtin_programs.configure(target=...)</code> binds the built-in programs to a concrete target. The DDR4 and HBM2 bundles expose the same calls for row writes, row reads, and RowHammer programs. Target-specific details such as HBM channel selection are added during program construction.

Building a program does not touch hardware. It returns a FinalProgram that can be held in memory, inspected, reused, and submitted later.


In [ ]:
BANK = 0
ROW = 0
PATTERN = 0xDEADBEEF

ddr4_programs = drambender.builtin_programs.configure(target=DDR4_TARGET)
hbm2_programs = drambender.builtin_programs.configure(target=HBM2_TARGET)

ddr4_pattern = (PATTERN,) * DDR4_TARGET.words_per_cacheline
hbm2_pattern = (PATTERN,) * HBM2_TARGET.words_per_cacheline

write_program = ddr4_programs.write_row(BANK, ROW, ddr4_pattern)
read_program = ddr4_programs.read_row(BANK, ROW)

hbm_write_program = hbm2_programs.write_row(BANK, ROW, hbm2_pattern)
hbm_read_program = hbm2_programs.read_row(BANK, ROW)

print(
    "DDR4 instructions:",
    write_program.instruction_count,
    "write,",
    read_program.instruction_count,
    "read",
)
print(
    "HBM2 instructions:",
    hbm_write_program.instruction_count,
    "write,",
    hbm_read_program.instruction_count,
    "read",
)


## 3. Inspect, dry-run, and trace a program

These tools run entirely on the CPU:

- <code>str(program)</code> prints the decoded instruction listing.
- <code>program.dry_run(max_instructions)</code> executes the program in the software VM and reports cycles, instruction counts, branches, registers, and DRAM command counts.
- <code>program.trace_dram_commands()</code> returns timestamped DRAM command events.

Labels in the listing mark branch targets; they are not uploaded instructions.


In [ ]:
listing_lines = str(read_program).splitlines()
print("\n".join(listing_lines[:30]))
if len(listing_lines) > 30:
    print(f"... {len(listing_lines) - 30} more lines")

result = read_program.dry_run(max_instructions=100_000)
print("\nDry-run result")
print(result)
print("DRAM command counts:", result.dram_cmd_counts)

trace = read_program.trace_dram_commands(max_instructions=100_000)
assert not trace.truncated
print("\nFirst DRAM command events")
for event in trace.events[:12]:
    print(
        f"t={event.time_ns:7.1f} ns, "
        f"delta={event.delta_ns:7.1f} ns, "
        f"command={event.command}, bank={event.bank}"
    )


The trace time is measured from program start. Each event delta is the elapsed time since the preceding DRAM command.

<code>trace.summarize_timings()</code> reports observed tRCD, tRAS, and tRP intervals. It does not compare them with a device specification and does not cover every timing constraint. The summary groups by bank, so use it as a quick check for traces that address one memory target at a time. Compare the reported intervals with the requirements for the module under test.


In [ ]:
print(trace.summarize_timings())


## 4. Execute a DDR4 read/write program

The hardware workflow is:

1. Open one endpoint with a target, complete PCI BDF, and XDMA channel.
2. Establish a clean session with <code>full_reset()</code>.
3. Submit one program or a list of programs with <code>execute()</code>.
4. Copy the expected number of ordered readback bytes into a writable, C-contiguous NumPy buffer with <code>receive_into()</code>.
5. Call <code>synchronize()</code> to wait for the active readback session and surface asynchronous errors.

The context manager closes the handle and releases endpoint ownership. It does not itself reset the FPGA. Opening the board prints its BoardConfig and reminds you that the programmed bitstream must match. The message reports API assumptions; it does not identify the bitstream automatically.


In [ ]:
if not RUN_HARDWARE:
    print("Skipped DDR4 hardware execution.")
else:
    readback = np.empty(DDR4_ROW_BYTES // 4, dtype=np.uint32)

    with open_board(
        DDR4_TARGET,
        pci_bdf=PCI_BDF,
        xdma_channel=XDMA_CHANNEL,
        host_interface=HostInterface.XDMA,
    ) as board:
        board.full_reset()
        board.execute([write_program, read_program])
        board.receive_into(readback, timeout=None)
        board.synchronize()

    expected = np.full_like(readback, PATTERN)
    mismatches = int(np.count_nonzero(readback != expected))
    assert mismatches == 0, f"{mismatches} mismatched words"
    print(f"PASS: {readback.size} words matched 0x{PATTERN:08x}")


### Reset and interruption behavior

Use <code>reset_fpga()</code> for a normal synchronized logic reset. Use <code>full_reset()</code> to cancel active readback, reset FPGA logic, drain stale host data, and clear queued software data.

A <code>receive_into(..., timeout=None)</code> call waits without a deadline, including across long retention intervals. A finite receive timeout, an asynchronous readback error, or Ctrl+C during a main-thread Python receive or synchronization wait triggers a full reset before the exception is raised. If a process is killed or intentionally abandons in-flight work, the next process should begin with <code>full_reset()</code>.


## 5. RowHammer with built-in programs

A Row object applies an explicitly selected physical-to-logical row mapping and carries a write pattern. The linear mapping is the identity mapping. Set <code>ROW_MAPPING</code> to the mapping used by the experiment. The example pairs consecutive physical row IDs; adjust this relation when the module uses another physical adjacency.

Scalar patterns use identity bitline and DQ mappings. Experiments that need explicit bitline or DQ transformations can pass a configured DataPattern.

The small program below is built and dry-run on the CPU. The hardware sweep follows the same <code>RUN_HARDWARE</code> opt-in as the read/write examples.


In [ ]:
from drambender.rows import Row, available_mappings

START_ROW = 81
victim = Row(
    physical_id=START_ROW,
    row_mapping=ROW_MAPPING,
    data_pattern=0x00000000,
)
aggressor = Row(
    physical_id=START_ROW + 1,
    row_mapping=ROW_MAPPING,
    data_pattern=0xFFFFFFFF,
)

hammer_preview = ddr4_programs.single_sided_rowhammer(
    BANK,
    aggressor.logical_id,
    hammer_count=8,
)
hammer_preview_result = hammer_preview.dry_run(max_instructions=10_000)

print("Available row mappings:", available_mappings())
print(
    f"victim physical={victim.physical_id}, logical={victim.logical_id}; "
    f"aggressor physical={aggressor.physical_id}, logical={aggressor.logical_id}"
)
print(hammer_preview_result)


In [ ]:
HAMMER_COUNT = 500_000
NUM_VICTIMS = 5
ROW_WORDS = DDR4_ROW_BYTES // 4

if not RUN_HARDWARE:
    print("Skipped the built-in RowHammer hardware sweep.")
else:
    total_bitflips = 0

    with open_board(
        DDR4_TARGET,
        pci_bdf=PCI_BDF,
        xdma_channel=XDMA_CHANNEL,
        host_interface=HostInterface.XDMA,
    ) as board:
        board.full_reset()

        for offset in range(NUM_VICTIMS):
            victim_physical = START_ROW + offset
            aggressor_physical = victim_physical + 1

            victim = Row(
                physical_id=victim_physical,
                row_mapping=ROW_MAPPING,
                data_pattern=0x00000000,
            )
            aggressor = Row(
                physical_id=aggressor_physical,
                row_mapping=ROW_MAPPING,
                data_pattern=0xFFFFFFFF,
            )

            board.execute([
                ddr4_programs.write_row(
                    BANK, victim.logical_id, victim.write_pattern
                ),
                ddr4_programs.write_row(
                    BANK, aggressor.logical_id, aggressor.write_pattern
                ),
                ddr4_programs.single_sided_rowhammer(
                    BANK, aggressor.logical_id, HAMMER_COUNT
                ),
                ddr4_programs.read_row(BANK, victim.logical_id),
            ])

            readback = np.empty(ROW_WORDS, dtype=np.uint32)
            board.receive_into(readback, timeout=None)
            board.synchronize()

            row_pattern = np.asarray(victim.write_pattern, dtype=np.uint32)
            expected = np.tile(row_pattern, ROW_WORDS // row_pattern.size)
            mask = readback ^ expected
            bitflips = int(
                np.unpackbits(mask.view(np.uint8), bitorder="little").sum()
            )
            total_bitflips += bitflips
            print(
                f"victim {victim_physical:5d}, "
                f"aggressor {aggressor_physical:5d}: "
                f"{bitflips} bit flips"
            )

    print(
        f"Total: {total_bitflips} bit flips across "
        f"{NUM_VICTIMS} victim rows"
    )


Readback is an ordered byte stream. The caller chooses how many bytes to copy into each buffer, so one receive call can consume part of a program's output or data from several queued programs. Buffer sizes must be multiples of four bytes.

Derive readback sizes from the target instead of hard-coding one row size. The explicit DDR4 target above yields 8192 bytes. The explicit HBM2 target yields 2048 raw bytes per row read.


## 6. Write a custom program with ProgramBuilder

ProgramBuilder emits DRAM Bender instructions:

1. Create it with an explicit DDR4Target or concrete HBM2 target.
2. Load scalar registers with operations such as <code>LI</code> and <code>ADDI</code>.
3. Copy scalar write values into the 16-lane wide write-data register with <code>LDWD</code>.
4. Issue DRAM commands with exact <code>DRAM</code> packing or a timed <code>DRAMSEQ</code>.
5. Add sleeps, labels, and branches.
6. Call <code>conclude()</code> to resolve control flow, append termination, and insert read-count metadata for the FPGA readback engine.

DRAM Bender has 16 scalar registers. Registers 0 through 6 have conventional names: CASR, BASR, RASR, CAR, BAR, RAR, and PATTERN_REG. PATTERN_REG is a scalar staging register; LDWD copies it into the separate wide write-data lanes. Nine additional registers can be allocated by name. ProgramBuilder does not spill registers.


Top-level slot-hiding calls such as <code>p.PRE()</code> and <code>p.ACT()</code> are intentionally forbidden. Use exact packing:

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())

For a multi-command timed sequence, delays are mini-slots:

    p.DRAMSEQ(
        PRE("BAR", delay=12),
        ACT("BAR", "RAR", delay=11),
        ALIGN(),
    )

The custom program below writes one cacheline. It uses exact packing so the command position remains visible.


In [ ]:
def build_write_cacheline(
    bank: int,
    row: int,
    column: int,
    pattern: int,
) -> FinalProgram:
    p = ProgramBuilder(target=DDR4_TARGET)
    p.LI(bank, "BAR")
    p.LI(row, "RAR")
    p.LI(column, "CAR")
    p.LI(DDR4_TARGET.column_stride, "CASR")

    p.LI(pattern, "PATTERN_REG")
    for lane in range(DDR4_TARGET.words_per_cacheline):
        p.LDWD("PATTERN_REG", lane)

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(9)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


custom_write = build_write_cacheline(
    bank=BANK,
    row=5,
    column=0,
    pattern=0xCAFEBABE,
)
print(custom_write.dry_run(max_instructions=10_000))
print(custom_write.trace_dram_commands().summarize_timings())


In [ ]:
CHECK_ROW = 5
CHECK_PATTERN = 0xCAFEBABE

if not RUN_HARDWARE:
    print("Skipped custom-program hardware execution.")
else:
    readback = np.empty(DDR4_ROW_BYTES // 4, dtype=np.uint32)

    with open_board(
        DDR4_TARGET,
        pci_bdf=PCI_BDF,
        xdma_channel=XDMA_CHANNEL,
        host_interface=HostInterface.XDMA,
    ) as board:
        board.full_reset()
        board.execute([
            ddr4_programs.write_row(
                BANK,
                CHECK_ROW,
                (0,) * DDR4_TARGET.words_per_cacheline,
            ),
            build_write_cacheline(
                BANK,
                CHECK_ROW,
                0,
                CHECK_PATTERN,
            ),
            ddr4_programs.read_row(BANK, CHECK_ROW),
        ])
        board.receive_into(readback, timeout=None)
        board.synchronize()

    first_cacheline = readback[:DDR4_TARGET.words_per_cacheline]
    remaining_words = readback[DDR4_TARGET.words_per_cacheline:]

    assert np.all(first_cacheline == CHECK_PATTERN)
    assert np.all(remaining_words == 0)
    print("PASS: custom cacheline write")


## 7. JIT-compiled program templates

The <code>@program_template</code> decorator traces a builder function, emits a native C++ plugin, compiles it, and caches the result. Later calls that vary only integer scalar arguments reuse the loaded specialization and patch those values. Non-integer arguments, including tuples and lists, are specialization inputs; changing them creates another specialization.

If the native JIT environment is unavailable, supported templates can fall back to interpreted construction. Unsupported template operations raise a compilation error instead of silently changing behavior. Use the run statistics to see what happened on the local system.


In [ ]:
@program_template
def build_write_cacheline_jit(
    bank: int,
    row: int,
    column: int,
    pattern: int,
) -> FinalProgram:
    p = ProgramBuilder(target=DDR4_TARGET)
    p.LI(bank, "BAR")
    p.LI(row, "RAR")
    p.LI(column, "CAR")
    p.LI(DDR4_TARGET.column_stride, "CASR")

    p.LI(pattern, "PATTERN_REG")
    for lane in range(DDR4_TARGET.words_per_cacheline):
        p.LDWD("PATTERN_REG", lane)

    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(9)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


In [ ]:
from drambender.api.jit import (
    get_jit_cache_dir,
    get_last_template_run_stats,
)

first_jit_program = build_write_cacheline_jit(
    BANK, 5, 0, 0x11111111
)
first_stats = get_last_template_run_stats()
assert first_stats is not None

second_jit_program = build_write_cacheline_jit(
    BANK, 6, 0, 0x22222222
)
second_stats = get_last_template_run_stats()
assert second_stats is not None

print("JIT cache:", get_jit_cache_dir())
print(
    "First call:",
    first_stats.mode,
    "cache_hit=",
    first_stats.cache_hit,
    "total_s=",
    f"{first_stats.total_s:.6f}",
)
print(
    "Second call:",
    second_stats.mode,
    "cache_hit=",
    second_stats.cache_hit,
    "total_s=",
    f"{second_stats.total_s:.6f}",
)


JIT diagnostics and cache controls live in <code>drambender.api.jit</code>:

- <code>get_last_template_run_stats()</code> reports the most recent construction path and timing.
- <code>get_jit_cache_dir()</code> reports the active on-disk cache.
- <code>set_jit_cache_dir(path)</code> selects another cache directory.
- <code>clear_template_caches()</code> clears in-memory specializations.
- <code>clear_template_caches(clear_disk=True)</code> also removes the active on-disk cache.


## 8. Advanced composition: custom hammer loop and built-in setup

One execute call can run an ordered sequence of built-in programs and custom JIT-generated programs. The custom template below hammers one aggressor row and then reads one victim row. Built-in programs initialize the two rows before it runs.

The preview uses a small hammer count so that dry-run and trace output remain bounded. A large count is used only inside the explicitly enabled hardware cell.


In [ ]:
@program_template
def build_hammer_and_read(
    bank: int,
    victim_row: int,
    aggressor_row: int,
    hammer_count: int,
) -> FinalProgram:
    p = ProgramBuilder(target=DDR4_TARGET)
    p.alloc_reg("NUM_HMR")
    p.alloc_reg("HMR_COUNTER")

    p.LI(bank, "BAR")
    p.LI(DDR4_TARGET.column_stride, "CASR")
    p.LI(aggressor_row, "RAR")
    p.LI(0, "HMR_COUNTER")
    p.LI(hammer_count, "NUM_HMR")

    p.LABEL("HAMMER")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.ADDI("HMR_COUNTER", 1, "HMR_COUNTER")
    p.DRAM(NOP(), NOP(), NOP(), ACT("BAR", "RAR"))
    p.BL("HMR_COUNTER", "NUM_HMR", "HAMMER")

    p.LI(victim_row, "RAR")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.LI(0, "CAR")
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    for _ in range(DDR4_TARGET.cachelines_per_row):
        p.DRAM(RD("BAR", "CAR", icar=1), NOP(), NOP(), NOP())
        p.SLEEP(1)
    p.SLEEP(4)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)
    return p.conclude()


In [ ]:
preview_program = build_hammer_and_read(
    bank=BANK,
    victim_row=START_ROW,
    aggressor_row=START_ROW + 1,
    hammer_count=8,
)

preview_result = preview_program.dry_run(max_instructions=100_000)
preview_trace = preview_program.trace_dram_commands(
    max_instructions=100_000
)
assert not preview_trace.truncated

hammer_act_times = [
    event.time_ns
    for event in preview_trace.events
    if event.command == "ACT"
][:8]

print(preview_result)
print(
    "Consecutive hammer ACT intervals (ns):",
    np.diff(hammer_act_times),
)


In [ ]:
ADVANCED_HAMMER_COUNT = 500_000

if not RUN_HARDWARE:
    print("Skipped advanced RowHammer hardware execution.")
else:
    victim = Row(
        physical_id=START_ROW,
        row_mapping=ROW_MAPPING,
        data_pattern=0x00000000,
    )
    aggressor = Row(
        physical_id=START_ROW + 1,
        row_mapping=ROW_MAPPING,
        data_pattern=0xFFFFFFFF,
    )

    program = build_hammer_and_read(
        bank=BANK,
        victim_row=victim.logical_id,
        aggressor_row=aggressor.logical_id,
        hammer_count=ADVANCED_HAMMER_COUNT,
    )

    with open_board(
        DDR4_TARGET,
        pci_bdf=PCI_BDF,
        xdma_channel=XDMA_CHANNEL,
        host_interface=HostInterface.XDMA,
    ) as board:
        board.full_reset()
        board.execute([
            ddr4_programs.write_row(
                BANK, victim.logical_id, victim.write_pattern
            ),
            ddr4_programs.write_row(
                BANK, aggressor.logical_id, aggressor.write_pattern
            ),
            program,
        ])
        readback = np.empty(ROW_WORDS, dtype=np.uint32)
        board.receive_into(readback, timeout=None)
        board.synchronize()

    row_pattern = np.asarray(victim.write_pattern, dtype=np.uint32)
    expected = np.tile(row_pattern, ROW_WORDS // row_pattern.size)
    mask = readback ^ expected
    bitflips = int(
        np.unpackbits(mask.view(np.uint8), bitorder="little").sum()
    )
    print(f"Observed {bitflips} bit flips")


## 9. Retention experiments

Both common retention workflows use the same API.

For a delay inside one DRAM Bender program, add one or more SLEEP instructions. Readback framing is independent of idle time, and <code>receive_into(timeout=None)</code> can wait across a long silent interval.

For a host-controlled delay, use two programs:

    board.execute(write_program)
    board.synchronize()
    time.sleep(retention_seconds)
    board.execute(read_program)
    board.receive_into(readback, timeout=None)
    board.synchronize()

Wait for the write program before starting the host delay. Configure automatic refresh with <code>board.set_aref(True)</code> or <code>board.set_aref(False)</code> according to the experiment.


## 10. Next steps

- Built-in programs: configure a target and use <code>read_row</code>, <code>write_row</code>, <code>single_sided_rowhammer</code>, or <code>double_sided_rowhammer</code>. See [python/drambender/builtin_programs/](../python/drambender/builtin_programs/).
- Row mappings: inspect <code>drambender.rows.available_mappings()</code> and the exported <code>linear</code>, <code>sa0</code>, and <code>mi1</code> mappings.
- Pattern mappings: see [python/drambender/patterns/](../python/drambender/patterns/).
- Low-level program code: start at [ProgramBuilder](../python/drambender/api/program/builder.py) and the [instruction factories](../python/drambender/api/program/instructions.py).
- C++ API: see [examples/read_write.cpp](read_write.cpp) and [examples/single_sided_rowhammer.cpp](single_sided_rowhammer.cpp).
- General setup and hardware design: return to the [repository README](../README.md).

Every hardware cell uses a context manager, so endpoint ownership is released when the cell exits. Begin a new session with <code>full_reset()</code> after an unclean process exit or abandoned in-flight program.
